# Shift-Scan: real end-to-end run (corrected)

Runs the five real experiments on the project zip: **decay calibration, teacher evaluation, QAT student training, student evaluation, GPU kernel tests**.

**What this version fixes compared with the earlier notebook**

| # | Problem in the earlier version | Fix here |
|---|---|---|
| 1 | The manifest wrote `"passed"` for every stage as a fixed string | Every status comes from the real return code, the expected output files and a required text check |
| 2 | No teacher evaluation ran, yet the manifest said `teacher_eval: passed` | A real teacher stage runs with the same protocol as the student, and the comparison fails loudly if it is missing |
| 3 | Training-time eval used `--val-seqs 8` (about 4k tokens), which is far too noisy | `VAL_SEQS = 64` (32k tokens) and a warning if it is set low again |
| 4 | Near-chance downstream scores were not flagged | The comparison table gives chance level, the teacher gap and a 2-standard-error check against chance |
| 5 | Kernel speed was never summarised (the Triton ternary GEMV is slower than FP16 here) | Slowdown ratios are parsed and a speed claim is blocked unless the data supports it |
| 6 | Duplicate cells, hard-coded paths, an unexplained inline rewrite of `calibrate_decay.py` | One config cell, auto-detected project path, and the rewrite is a safe, documented, reversible patch |
| 7 | A hung process after `done ->` would block the cell forever | A watchdog ends the process shortly after `done ->` and records that it did |
| 8 | Results lived only in `/content` | Optional copy to Drive, plus stage-level resume |

**How to run:** set GPU runtime, edit only the config cell, then *Run all*. To run another seed or variant, change `SEED` / `VARIANT` and run all again; outputs are kept in separate folders. Files from this protocol are tagged `PROTOCOL` and must **not** be mixed into the 10M-token, 305-step pilot table.

In [ ]:
# ===== 1. CONFIG: the only cell you normally edit =====
SEED = 2
VARIANT = "harpq"        # nearest | harpq | dyadic2 | fp_decay   (fp_decay = ternary weights, unquantized exp decay)
PROTOCOL = "fast2M"      # tag in every output name; do NOT mix with the 10M-token / 305-step pilot results

# calibration
CAL_TOKENS, CAL_SEQ_LEN, CAL_HORIZON, MAX_COEFFICIENTS = 200_000, 512, 128, 20_000

# QAT training
QAT_TOKENS, QAT_SEQ_LEN, QAT_ACCUM, QAT_LR = 2_000_000, 512, 4, 1e-4
VAL_SEQS = 64            # was 8 (about 4k tokens, far too noisy); 64 x 512 = 32k tokens
EVAL_EVERY, SAVE_EVERY, LOG_EVERY = 100, 250, 10      # SAVE_EVERY=50 wrote a 2 GB checkpoint 20 times

# evaluation
EVAL_LIMIT = None        # None = full lm-eval tasks (no sample cap)
RUN_TEACHER = True       # needed for the comparison table

# housekeeping
SAVE_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/shift_scan_runs/real_runs"
FORCE = False            # True = redo stages that already finished
REQUIRE_GPU = True
PIP_INSTALL = True
DONE_GRACE_S = 120       # kill the trainer this long after it prints "done ->" if it has not exited
POLL_S = 5
PROJECT_DIR = None       # None = auto-detect, or upload the zip

VARIANT_FLAGS = {
    "nearest":  ["--decay", "shift", "--decay-group-size", "1", "--decay-quantizer", "nearest"],
    "harpq":    ["--decay", "shift", "--decay-group-size", "1", "--decay-quantizer", "horizon",
                 "--decay-horizon", "128", "--decay-lag-weight", "uniform"],
    "dyadic2":  ["--decay", "shift", "--decay-group-size", "1", "--decay-quantizer", "dyadic",
                 "--dyadic-terms", "2", "--decay-horizon", "128"],
    "fp_decay": ["--decay", "exp"],
}
assert VARIANT in VARIANT_FLAGS, f"VARIANT must be one of {list(VARIANT_FLAGS)}"
if VAL_SEQS * QAT_SEQ_LEN < 16_000:
    print(f"WARNING: training-time eval uses only {VAL_SEQS * QAT_SEQ_LEN} tokens; its PPL will be noisy. Do not quote it.")


In [ ]:
# ===== 2. Setup: project, GPU, folders =====
import os, sys, json, time, shutil, zipfile, hashlib, random, pathlib, platform, subprocess, signal, re, ast
import numpy as np
try:
    import torch
except ImportError:
    torch = None
try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

if PIP_INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "datasets", "transformers",
                    "huggingface_hub", "tqdm", "triton", "lm-eval", "pandas"], check=False)

if SAVE_TO_DRIVE and IN_COLAB:
    drive.mount("/content/drive")

ZIP_SHA256 = None
def find_project():
    global ZIP_SHA256
    if PROJECT_DIR:
        p = pathlib.Path(PROJECT_DIR); assert (p / "train_qat.py").exists(), f"train_qat.py not in {p}"
        return p
    root = pathlib.Path("/content/shift_scan_advanced_project")
    hits = list(root.rglob("train_qat.py")) if root.exists() else []
    if not hits:                                        # first run: upload the zip once
        assert IN_COLAB, "Set PROJECT_DIR outside Colab"
        up = files.upload()
        zips = [n for n in up if n.lower().endswith(".zip")]
        assert zips, "Upload shift_scan_advanced_project.zip"
        zname = next((n for n in zips if "shift_scan_advanced_project" in n.lower()), zips[0])
        ZIP_SHA256 = sha256_file(zname)
        if root.exists(): shutil.rmtree(root)
        root.mkdir(parents=True)
        with zipfile.ZipFile(zname) as z: z.extractall(root)
        hits = list(root.rglob("train_qat.py"))
        assert hits, "Wrong archive: train_qat.py missing"
    else:                                               # reuse extracted project, hash the zip if still present
        for z in pathlib.Path("/content").glob("shift_scan_advanced_project*.zip"):
            ZIP_SHA256 = sha256_file(z); break
    return hits[0].parent

PROJECT = find_project()
os.chdir(PROJECT)
need = ["ssm_core.py", "calibrate_decay.py", "eval_lm.py", "train_qat.py", "kernels.py", "tests.py"]
missing = [f for f in need if not (PROJECT / f).exists()]
assert not missing, f"project files missing: {missing}"

# GPU: must exist and must be free (a stopped cell can leave a trainer running and holding memory)
def gpu_used_mib():
    out = sh("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -i 0")
    return int(out.split()[0]) if out else 0

def kill_strays():
    for pat in ("train_qat.py", "eval_lm.py", "calibrate_decay.py"):
        for pid in sh(f"pgrep -f {pat}").split():
            if int(pid) != os.getpid():
                try: os.kill(int(pid), signal.SIGKILL)
                except ProcessLookupError: pass

GPU_NAME = "none"
if REQUIRE_GPU:
    assert torch is not None and torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
    GPU_NAME = sh("nvidia-smi --query-gpu=name --format=csv,noheader -i 0")
    if gpu_used_mib() > 1500:
        print("GPU already holds", gpu_used_mib(), "MiB: ending leftover project processes")
        kill_strays(); time.sleep(8)
    assert gpu_used_mib() < 1500, "GPU still busy: Runtime > Restart session, then rerun"

RUN_NAME = f"real_{VARIANT}_{PROTOCOL}_s{SEED}"
OUT = PROJECT / "results" / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

ENV = {"python": sys.version.split()[0], "torch": getattr(torch, "__version__", None),
       "cuda": getattr(getattr(torch, "version", None), "cuda", None), "gpu": GPU_NAME,
       "platform": platform.platform()}
print({"run": RUN_NAME, "project": str(PROJECT), "out": str(OUT), **ENV, "zip_sha256": ZIP_SHA256})


In [ ]:
# ===== 3. Stage runner with watchdog and honest status =====
STATUS_PATH = OUT / "stage_status.json"
STATUS = json.loads(STATUS_PATH.read_text()) if STATUS_PATH.exists() else {}

def save_status():
    STATUS_PATH.write_text(json.dumps(STATUS, indent=2))

def read(p):
    p = pathlib.Path(p)
    return p.read_text(errors="ignore") if p.exists() else ""

def _tail(log, pos, last_prog, t0, final=False):
    """Print new log lines. tqdm progress lines are throttled to one per minute."""
    with open(log, "rb") as f:
        f.seek(pos); data = f.read(); pos += len(data)
    for ln in re.split(r"[\r\n]+", data.decode("utf-8", "replace")):
        ln = ln.rstrip()
        if not ln: continue
        is_prog = "%|" in ln
        if is_prog and not final and time.time() - last_prog < 60: continue
        if is_prog: last_prog = time.time()
        print(f"[{(time.time() - t0) / 60:6.1f} min] {ln[:240]}", flush=True)
    return pos, last_prog

def stage(name, cmd, log, cwd=None, done_marker=None, outputs=(), require_text=None, fatal=True):
    """Run one stage. ok = return code 0 AND expected files exist AND required text is in the log."""
    cwd = cwd or PROJECT
    prev = STATUS.get(name)
    if prev and prev.get("ok") and not FORCE and all(pathlib.Path(p).exists() for p in outputs):
        print(f"[skip] {name}: already finished ({prev['minutes']} min)"); return prev
    env = dict(os.environ, PYTHONUNBUFFERED="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    cmd = [str(c) for c in cmd]
    print("=" * 70); print("STAGE:", name); print("COMMAND:", " ".join(cmd)); print("=" * 70, flush=True)
    t0 = time.time(); watchdog = False; done_t = None
    with open(log, "wb") as lf:
        p = subprocess.Popen(cmd, cwd=str(cwd), stdout=lf, stderr=subprocess.STDOUT, env=env, start_new_session=True)
        pos, last_prog = 0, 0.0
        try:
            while True:
                time.sleep(POLL_S)
                finished = p.poll() is not None
                pos, last_prog = _tail(log, pos, last_prog, t0, final=finished)
                if finished: break
                if done_marker and done_marker in read(log):
                    done_t = done_t or time.time()
                    if time.time() - done_t > DONE_GRACE_S:        # finished but will not exit
                        os.killpg(p.pid, signal.SIGKILL); p.wait(); watchdog = True; break
        except KeyboardInterrupt:
            os.killpg(p.pid, signal.SIGKILL); raise
    rc = 0 if watchdog else p.returncode
    text = read(log)
    missing = [str(o) for o in outputs if not pathlib.Path(o).exists()]
    why = []
    if rc != 0: why.append(f"return code {rc}")
    if missing: why.append(f"missing outputs {missing}")
    if require_text and require_text not in text: why.append(f"log lacks required text {require_text!r}")
    rec = {"ok": not why, "rc": rc, "minutes": round((time.time() - t0) / 60, 2),
           "watchdog_ended_process": watchdog, "log": str(log), "problems": why}
    STATUS[name] = rec; save_status()
    print(f"\n[{name}] ok={rec['ok']} rc={rc} {rec['minutes']} min" + (" (watchdog ended the hung process)" if watchdog else ""))
    if why:
        print("PROBLEMS:", "; ".join(why)); print(text[-1800:])
        if fatal: raise RuntimeError(f"stage {name} failed: {'; '.join(why)}")
    return rec


In [ ]:
# ===== 4. Repository tests =====
stage("repo_tests", [sys.executable, "tests.py"], OUT / "tests.log", require_text="all tests passed")


## 5. Calibration: fix the token-count bug, then run it on real FineWeb-Edu

The `calibrate_decay.py` in the zip stopped reading text once the coefficient cap was reached, so the earlier calibration reports said `tokens_seen = 7168` even though 200,000 tokens were requested. The patch below makes it sample coefficients across the whole validation stream (this is the rewrite you had inline before). It is applied only if needed, keeps a `.orig` backup, and restores the original automatically if the patched script does not start.

In [ ]:
FIXED_CALIBRATE = r'''
"""
SHIFT-SCAN: FineWeb-EDU Power-of-Two Decay Calibration

Compares:
    1. Nearest one-step power-of-two exponent
    2. Horizon-aware HARPQ exponent

Samples decay coefficients across the entire validation stream.
The coefficient cap limits sampling only, NOT token processing.
"""

import argparse
import json
import os
import math

import torch

from ssm_core import load_pretrained, QuantConfig, get_tokenizer
from retention import (
    exponent_grid,
    horizon_aware_exponent,
    retention_error_report,
)


# ============================================================
# ARGUMENTS
# ============================================================

def get_args():
    p = argparse.ArgumentParser()

    p.add_argument(
        "--pretrained",
        default="state-spaces/mamba2-130m",
    )

    p.add_argument(
        "--dataset",
        default="HuggingFaceFW/fineweb-edu",
    )

    p.add_argument(
        "--subset",
        default="sample-10BT",
    )

    p.add_argument(
        "--val-modulo",
        type=int,
        default=100,
    )

    p.add_argument(
        "--tokens",
        type=int,
        default=200000,
    )

    p.add_argument(
        "--seq-len",
        type=int,
        default=1024,
    )

    p.add_argument(
        "--k-min",
        type=int,
        default=1,
    )

    p.add_argument(
        "--k-max",
        type=int,
        default=14,
    )

    p.add_argument(
        "--horizon",
        type=int,
        default=128,
    )

    p.add_argument(
        "--weight",
        choices=["uniform", "log", "early", "late"],
        default="uniform",
    )

    p.add_argument(
        "--dtype",
        choices=["auto", "bf16", "fp16"],
        default="auto",
    )

    p.add_argument(
        "--max-coefficients",
        type=int,
        default=20000,
    )

    p.add_argument(
        "--out",
        required=True,
    )

    return p.parse_args()


# ============================================================
# UNIFORM COEFFICIENT SAMPLING
# ============================================================

def sample_uniform_indices(num_values, quota, device):
    """
    Select approximately uniform indices from a flattened tensor.
    """

    if quota <= 0 or num_values <= 0:
        return torch.empty(
            0,
            dtype=torch.long,
            device=device,
        )

    if quota >= num_values:
        return torch.arange(
            num_values,
            dtype=torch.long,
            device=device,
        )

    indices = torch.linspace(
        0,
        num_values - 1,
        steps=quota,
        device=device,
    )

    indices = indices.round().long()

    return indices.unique()


# ============================================================
# MAIN
# ============================================================

@torch.no_grad()
def main():

    a = get_args()

    from datasets import load_dataset

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Decay calibration requires CUDA."
        )

    if a.tokens <= 0:
        raise ValueError("--tokens must be positive")

    if a.seq_len <= 0:
        raise ValueError("--seq-len must be positive")

    if a.max_coefficients <= 0:
        raise ValueError(
            "--max-coefficients must be positive"
        )

    if a.val_modulo <= 0:
        raise ValueError(
            "--val-modulo must be positive"
        )

    if a.k_min > a.k_max:
        raise ValueError(
            "--k-min must be <= --k-max"
        )

    if a.horizon <= 0:
        raise ValueError(
            "--horizon must be positive"
        )

    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    print("\nLoading tokenizer...", flush=True)

    tok = get_tokenizer()

    print(
        f"\nLoading model: {a.pretrained}",
        flush=True,
    )

    model, _ = load_pretrained(
        a.pretrained,
        QuantConfig(decay="exp"),
    )

    model = model.cuda().eval()

    if a.dtype == "auto":

        amp_dtype = (
            torch.bfloat16
            if torch.cuda.is_bf16_supported()
            else torch.float16
        )

    elif a.dtype == "bf16":

        amp_dtype = torch.bfloat16

    else:

        amp_dtype = torch.float16

    print(
        "Calibration dtype:",
        amp_dtype,
        flush=True,
    )

    # --------------------------------------------------------
    # DATASET
    # --------------------------------------------------------

    print(
        "\nLoading streaming dataset...",
        flush=True,
    )

    ds = load_dataset(
        a.dataset,
        name=a.subset,
        split="train",
        streaming=True,
    )

    ds = ds.filter(
        lambda ex, idx: idx % a.val_modulo == 0,
        with_indices=True,
    )

    # --------------------------------------------------------
    # EXPONENT GRID
    # --------------------------------------------------------

    grid = exponent_grid(
        a.k_min,
        a.k_max,
        1.0,
        device="cuda",
    )

    # --------------------------------------------------------
    # CALIBRATION STATE
    # --------------------------------------------------------

    chunks = []

    buf = []

    seen = 0

    coefficients_collected = 0

    target_sequences = math.ceil(
        a.tokens / a.seq_len
    )

    print(
        "\nEstimated sequence forwards:",
        target_sequences,
        flush=True,
    )

    print(
        "Coefficient sampling cap:",
        a.max_coefficients,
        flush=True,
    )

    print(
        "\nStarting calibration...",
        flush=True,
    )

    # --------------------------------------------------------
    # STREAMING CALIBRATION
    # --------------------------------------------------------

    for ex in ds:

        txt = ex.get("text", "")

        if not isinstance(txt, str) or not txt.strip():
            continue

        token_ids = tok(
            txt,
            add_special_tokens=False,
        )["input_ids"]

        token_ids = token_ids + [tok.eos_token_id]

        buf.extend(token_ids)

        while len(buf) >= a.seq_len + 1:

            # Stop ONLY after target tokens are processed.
            if seen >= a.tokens:
                break

            ids = torch.tensor(
                buf[:a.seq_len],
                device="cuda",
                dtype=torch.long,
            )[None, :]

            buf = buf[a.seq_len:]

            # ------------------------------------------------
            # FORWARD PASS
            # ------------------------------------------------

            model.set_track_decay(True)

            try:

                with torch.autocast(
                    "cuda",
                    dtype=amp_dtype,
                ):

                    model(ids)

                # Gather decay coefficients from every layer.
                layer_values = []

                for layer in model.backbone.layers:

                    decay_ref = (
                        layer.mixer.last_decay_ref
                    )

                    if decay_ref is None:
                        continue

                    values = (
                        decay_ref
                        .detach()
                        .float()
                        .reshape(-1)
                    )

                    if values.numel() > 0:
                        layer_values.append(values)

                if not layer_values:

                    raise RuntimeError(
                        "No decay coefficients were captured. "
                        "Check set_track_decay() and last_decay_ref."
                    )

                batch_values = torch.cat(
                    layer_values
                )

            finally:

                model.set_track_decay(False)

            # ------------------------------------------------
            # DISTRIBUTED COEFFICIENT SAMPLING
            # ------------------------------------------------

            remaining = (
                a.max_coefficients
                - coefficients_collected
            )

            if remaining > 0:

                sequences_remaining = max(
                    1,
                    target_sequences
                    - (seen // a.seq_len),
                )

                quota = math.ceil(
                    remaining / sequences_remaining
                )

                quota = min(
                    quota,
                    batch_values.numel(),
                )

                indices = sample_uniform_indices(
                    num_values=batch_values.numel(),
                    quota=quota,
                    device=batch_values.device,
                )

                sampled_values = (
                    batch_values[indices]
                    .detach()
                    .float()
                    .cpu()
                )

                # Clip in case unique rounding reduced
                # or otherwise altered the desired sample.
                if sampled_values.numel() > remaining:

                    sampled_values = (
                        sampled_values[:remaining]
                    )

                if sampled_values.numel() > 0:

                    chunks.append(sampled_values)

                    coefficients_collected += (
                        sampled_values.numel()
                    )

            # ------------------------------------------------
            # TOKEN COUNTER
            # ------------------------------------------------

            seen += a.seq_len

            # ------------------------------------------------
            # PROGRESS
            # ------------------------------------------------

            if (
                seen % (a.seq_len * 10) == 0
                or seen >= a.tokens
            ):

                print(
                    f"Tokens processed: {seen:,} / "
                    f"{a.tokens:,} | "
                    f"Coefficients sampled: "
                    f"{coefficients_collected:,} / "
                    f"{a.max_coefficients:,}",
                    flush=True,
                )

            # IMPORTANT:
            # Do NOT break when coefficient sampling cap
            # is reached. Continue processing tokens.

        if seen >= a.tokens:
            break

    # ========================================================
    # VALIDATE SAMPLING
    # ========================================================

    if seen == 0:

        raise RuntimeError(
            "No validation tokens were produced. "
            "Check the dataset and validation configuration."
        )

    if not chunks:

        raise RuntimeError(
            "No decay coefficients were captured."
        )

    a_ref = torch.cat(chunks)

    print(
        "\nCalibration stream completed.",
        flush=True,
    )

    print(
        "Actual tokens processed:",
        seen,
        flush=True,
    )

    print(
        "Actual coefficients collected:",
        a_ref.numel(),
        flush=True,
    )

    # --------------------------------------------------------
    # MOVE COEFFICIENT VECTOR TO GPU
    # --------------------------------------------------------

    a_ref = a_ref.clamp(
        1e-8,
        1 - 1e-8,
    )

    a_ref = a_ref.cuda()

    # ========================================================
    # HARPQ SCORING
    # ========================================================

    print(
        "\nComputing nearest and horizon-aware exponents...",
        flush=True,
    )

    (
        k_h,
        cost_h,
        k_near,
        cost_near,
    ) = horizon_aware_exponent(
        a_ref,
        grid,
        a.horizon,
        a.weight,
    )

    a_near = 1 - torch.exp2(-k_near)

    a_h = 1 - torch.exp2(-k_h)

    # ========================================================
    # REPORT
    # ========================================================

    nearest_report = retention_error_report(
        a_ref,
        a_near,
        a.horizon,
    )

    horizon_report = retention_error_report(
        a_ref,
        a_h,
        a.horizon,
    )

    nearest_mean = cost_near.mean().item()

    horizon_mean = cost_h.mean().item()

    relative_reduction = (
        1
        - horizon_mean
        / max(nearest_mean, 1e-12)
    )

    out = {

        "source": {

            "model": a.pretrained,

            "dataset": a.dataset,

            "subset": a.subset,

            "validation_rule": (
                f"document_index % "
                f"{a.val_modulo} == 0"
            ),

            "tokens_seen": seen,

        },

        "method": {

            "proposed_name": "HARPQ",

            "objective": (
                "weighted finite-horizon "
                "stationary retention-curve MSE"
            ),

            "horizon": a.horizon,

            "lag_weight": a.weight,

            "k_grid": [
                a.k_min,
                a.k_max,
            ],

            "calibration_dtype": str(
                amp_dtype
            ),

            "sampling_strategy": (
                "uniformly distributed coefficient "
                "sampling across processed sequences"
            ),

            "coefficient_cap": (
                a.max_coefficients
            ),

        },

        "coefficients": int(
            a_ref.numel()
        ),

        "fraction_exponents_changed_vs_nearest": (
            (k_h != k_near)
            .float()
            .mean()
            .item()
        ),

        "nearest_one_step": nearest_report,

        "horizon_aware": horizon_report,

        "objective_cost": {

            "nearest_mean": nearest_mean,

            "horizon_mean": horizon_mean,

            "relative_reduction": (
                relative_reduction
            ),

        },

        "k_histogram_nearest": (
            torch.bincount(
                k_near.long() - a.k_min,
                minlength=a.k_max - a.k_min + 1,
            )
            .cpu()
            .tolist()
        ),

        "k_histogram_horizon": (
            torch.bincount(
                k_h.long() - a.k_min,
                minlength=a.k_max - a.k_min + 1,
            )
            .cpu()
            .tolist()
        ),

        "warning": (
            "Calibration diagnostics only. This does not "
            "establish perplexity, speed, energy, or "
            "global novelty."
        ),

    }

    # ========================================================
    # SAVE
    # ========================================================

    os.makedirs(
        os.path.dirname(a.out) or ".",
        exist_ok=True,
    )

    with open(
        a.out,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            out,
            f,
            indent=2,
        )

    print(
        "\n" + "=" * 65,
        flush=True,
    )

    print(
        "CALIBRATION RESULTS",
        flush=True,
    )

    print(
        "=" * 65,
        flush=True,
    )

    print(
        json.dumps(
            out,
            indent=2,
        ),
        flush=True,
    )

    print(
        "\nSaved to:",
        a.out,
        flush=True,
    )


if __name__ == "__main__":

    main()
'''

CAL = PROJECT / "calibrate_decay.py"
if "sampling_strategy" in CAL.read_text():
    print("calibrate_decay.py already contains the sampling fix")
else:
    backup = CAL.with_suffix(".py.orig")
    if not backup.exists(): shutil.copy(CAL, backup)
    CAL.write_text(FIXED_CALIBRATE, encoding="utf-8")
    r = subprocess.run([sys.executable, str(CAL), "--help"], cwd=PROJECT, capture_output=True, text=True)
    if r.returncode != 0 or "--max-coefficients" not in r.stdout:
        shutil.copy(backup, CAL)
        print("Patched script did not start, original restored. Reason:\n", (r.stderr or r.stdout)[-600:])
    else:
        print("calibrate_decay.py patched (original saved as calibrate_decay.py.orig)")


In [ ]:
# ===== 5b. Run calibration =====
CAL_JSON = OUT / "fineweb_harpq_calibration.json"
rec = stage("fineweb_calibration",
            [sys.executable, "-u", PROJECT / "calibrate_decay.py",
             "--pretrained", "state-spaces/mamba2-130m", "--dataset", "HuggingFaceFW/fineweb-edu",
             "--subset", "sample-10BT", "--val-modulo", "100", "--tokens", CAL_TOKENS,
             "--seq-len", CAL_SEQ_LEN, "--horizon", CAL_HORIZON, "--max-coefficients", MAX_COEFFICIENTS,
             "--dtype", "fp16", "--out", CAL_JSON],
            OUT / "calibration.log", outputs=[CAL_JSON])

cal = json.loads(CAL_JSON.read_text())
seen = cal["source"]["tokens_seen"]
if seen < CAL_TOKENS:                                   # a shortfall is a failure, not a warning
    STATUS["fineweb_calibration"].update(ok=False, problems=[f"tokens_seen {seen} < target {CAL_TOKENS}"]); save_status()
    print(f"CALIBRATION INCOMPLETE: only {seen} of {CAL_TOKENS} tokens were processed (is the sampling patch applied?)")
n, h = cal["nearest_one_step"], cal["horizon_aware"]
print(f"tokens_seen {seen} | coefficients {cal['coefficients']} | exponents changed by HARPQ: {cal['fraction_exponents_changed_vs_nearest']:.2%}")
print(f"retention RMSE  nearest {n['retention_rmse']:.5f}  HARPQ {h['retention_rmse']:.5f}  "
      f"({(1 - h['retention_rmse'] / n['retention_rmse']):.2%} lower)")
print(f"objective cost reduction: {cal['objective_cost']['relative_reduction']:.2%}")
print("Note: this is a coefficient diagnostic. It does not show perplexity, speed or energy gains.")


## 6. Teacher evaluation (same protocol as the student)

The earlier notebook claimed this stage passed but never ran it. The student can only be judged next to the pretrained Mamba-2 evaluated with the same script, data and tasks.

In [ ]:
TEACHER_JSON = OUT / "teacher_fineweb_ppl.json"
if RUN_TEACHER:
    cmd = [sys.executable, "eval_lm.py", "--pretrained", "state-spaces/mamba2-130m",
           "--dataset", "HuggingFaceFW/fineweb-edu", "--subset", "sample-10BT", "--val-modulo", "100",
           "--ppl-length", "--dtype", "fp16", "--out", TEACHER_JSON]
    if EVAL_LIMIT is not None: cmd += ["--limit", EVAL_LIMIT]
    stage("teacher_eval", cmd, OUT / "teacher_eval.log", outputs=[TEACHER_JSON])
else:
    print("RUN_TEACHER is False: the comparison table and manifest will say the teacher is missing")


## 7. QAT student training

Only the variant flags change between `nearest`, `harpq`, `dyadic2` and `fp_decay`. The manifest records the exact command.

In [ ]:
CKPT = PROJECT / "runs" / RUN_NAME / "last.pt"
train_cmd = [sys.executable, "-u", "train_qat.py", "--name", RUN_NAME,
             "--pretrained", "state-spaces/mamba2-130m"] + VARIANT_FLAGS[VARIANT] + [
             "--ternary", "1", "--group-size", "128", "--activation-aware", "1",
             "--state-weight", "0.05", "--state-loss-type", "hybrid",
             "--tokens", QAT_TOKENS, "--seq-len", QAT_SEQ_LEN, "--micro-bs", "1", "--accum", QAT_ACCUM,
             "--lr", QAT_LR, "--dtype", "fp16", "--val-seqs", VAL_SEQS,
             "--eval-every", EVAL_EVERY, "--save-every", SAVE_EVERY, "--log-every", LOG_EVERY,
             "--seed", SEED, "--out", PROJECT / "runs"]

if VARIANT == "fp_decay":                               # make sure this flag value exists before spending an hour
    helptxt = subprocess.run([sys.executable, "train_qat.py", "--help"], cwd=PROJECT, capture_output=True, text=True).stdout
    assert "exp" in helptxt, "train_qat.py --help does not list 'exp' for --decay; check the choices before running fp_decay"

rec = stage("qat_training", train_cmd, OUT / "qat_training.log", done_marker="done ->", outputs=[CKPT])

evals = [ast.literal_eval(l[5:]) for l in read(OUT / "qat_training.log").splitlines() if l.startswith("EVAL ")]
print(f"\ntraining-time eval uses {VAL_SEQS * QAT_SEQ_LEN} tokens per check")
for e in evals[-5:]:
    print(f"  step {e['step']:>5}  student_ppl {e['student_ppl']:.2f}  state_mse {e['state_mse']:.4f}")
if len(evals) >= 3:
    last3 = [e["student_ppl"] for e in evals[-3:]]
    print(f"  last 3 checks: mean {np.mean(last3):.2f}, spread {max(last3) - min(last3):.2f}  (quote lm-eval results, not this)")


In [ ]:
# ===== 8. Student evaluation (same eval_lm.py protocol as the teacher) =====
STUDENT_JSON = OUT / "student_fineweb_ppl.json"
cmd = [sys.executable, "eval_lm.py", "--ckpt", CKPT, "--dataset", "HuggingFaceFW/fineweb-edu",
       "--subset", "sample-10BT", "--val-modulo", "100", "--ppl-length", "--dtype", "fp16", "--out", STUDENT_JSON]
if EVAL_LIMIT is not None: cmd += ["--limit", EVAL_LIMIT]
stage("student_eval", cmd, OUT / "student_eval.log", outputs=[STUDENT_JSON])
student = json.loads(STUDENT_JSON.read_text())
print("loss by position:", {k: round(v, 3) for k, v in student.get("loss_by_position", {}).items()})


In [ ]:
# ===== 9. GPU kernel tests and an honest speed summary =====
KERNEL_JSON = OUT / "kernel_test_status.json"
rec = stage("kernel_tests", [sys.executable, "kernels.py"], OUT / "kernels.log", outputs=[])
ktext = read(OUT / "kernels.log")
ratios = []
for line in ktext.splitlines():
    m = re.search(r"^\s*(.+?)\s+B=(\d+): fp16\s+([\d.]+)\s*\|\s*ternary-triton\s+([\d.]+)", line)
    if m:
        shape, b, fp, tr = m.group(1).strip(), int(m.group(2)), float(m.group(3)), float(m.group(4))
        ratios.append({"shape": shape, "batch": b, "fp16_us": fp, "ternary_us": tr, "ternary_over_fp16": round(tr / fp, 2)})
slower = [r for r in ratios if r["ternary_over_fp16"] > 1.0]
speed_claim_ok = bool(ratios) and not slower
for r in ratios:
    print(f"  {r['shape']:<28} B={r['batch']}: fp16 {r['fp16_us']:7.1f} us | ternary {r['ternary_us']:7.1f} us | {r['ternary_over_fp16']:>5.2f}x")
print("\nCorrectness:", "PASSED" if rec["ok"] else "FAILED")
print("Speed claim supported:", speed_claim_ok,
      "" if speed_claim_ok else f"-> the ternary kernel is slower than FP16 in {len(slower)} of {len(ratios)} cases; report correctness and memory savings only")
KERNEL_JSON.write_text(json.dumps({"correctness_passed": rec["ok"], "speed_claim_supported": speed_claim_ok,
                                   "microbenchmarks": ratios}, indent=2))


In [ ]:
# ===== 10. Teacher vs student, with chance level =====
import pandas as pd
assert TEACHER_JSON.exists(), "No teacher results: run section 6 (RUN_TEACHER=True). A student number alone proves nothing."
teacher = json.loads(TEACHER_JSON.read_text())["results"]
student = json.loads(STUDENT_JSON.read_text())["results"]

# (task, metric key, label, chance level or None, higher_is_better)
SPEC = [("wikitext", "word_perplexity,none", "wikitext word PPL", None, False),
        ("lambada_openai", "perplexity,none", "lambada PPL", None, False),
        ("lambada_openai", "acc,none", "lambada acc", 0.0, True),
        ("hellaswag", "acc_norm,none", "hellaswag acc_norm", 0.25, True),
        ("piqa", "acc_norm,none", "piqa acc_norm", 0.50, True),
        ("arc_easy", "acc_norm,none", "arc_easy acc_norm", 0.25, True),
        ("arc_challenge", "acc_norm,none", "arc_challenge acc_norm", 0.25, True),
        ("winogrande", "acc,none", "winogrande acc", 0.50, True)]
rows = []
for task, key, label, chance, higher in SPEC:
    t = teacher.get(task, {}).get(key); s = student.get(task, {}).get(key)
    if t is None or s is None: continue
    se = student[task].get(key.replace(",none", "_stderr,none"))
    row = {"metric": label, "teacher": round(t, 4), "student": round(s, 4)}
    if higher:
        row["student/teacher"] = round(s / t, 2) if t else None
        if chance is not None and isinstance(se, (int, float)) and chance > 0:
            row["chance"] = chance
            row["student above chance (2 SE)"] = bool(s - chance > 2 * se)
    else:
        row["student/teacher"] = round(s / t, 1)
    rows.append(row)
CMP = pd.DataFrame(rows)
CMP.to_csv(OUT / "teacher_vs_student.csv", index=False)
print(CMP.to_string(index=False))
near_chance = [r["metric"] for r in rows if r.get("student above chance (2 SE)") is False]
print("\nNot distinguishable from chance:", near_chance or "none")


In [ ]:
# ===== 11. Manifest: every status is computed, nothing is hard-coded =====
caveats = ["single seed per condition"]
if QAT_TOKENS < 10_000_000: caveats.append(f"pilot scale: {QAT_TOKENS:,} training tokens")
if VAL_SEQS * QAT_SEQ_LEN < 16_000: caveats.append("training-time eval used very few tokens")
if not (STATUS.get("fineweb_calibration", {}).get("ok")): caveats.append("calibration did not reach its token target")
if not TEACHER_JSON.exists(): caveats.append("no teacher baseline in this bundle")
if near_chance: caveats.append("not above chance: " + ", ".join(near_chance))
if not speed_claim_ok: caveats.append("ternary Triton kernel is not faster than FP16 in the microbenchmark; no speed claim")
caveats.append(f"protocol '{PROTOCOL}' differs from the 10M-token / 305-step pilots; do not pool the two")

artifacts = sorted(p for p in OUT.glob("*") if p.is_file() and p.name != "real_run_manifest.json")
manifest = {
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "run_name": RUN_NAME, "variant": VARIANT, "seed": SEED, "protocol": PROTOCOL,
    "source_zip_sha256": ZIP_SHA256, "project_dir": str(PROJECT), "environment": ENV,
    "stages": {k: {"ok": v["ok"], "return_code": v["rc"], "minutes": v["minutes"],
                   "watchdog_ended_process": v["watchdog_ended_process"], "problems": v["problems"]}
               for k, v in STATUS.items()},
    "all_stages_ok": all(v["ok"] for v in STATUS.values()),
    "settings": {"cal_tokens": CAL_TOKENS, "cal_seq_len": CAL_SEQ_LEN, "cal_horizon": CAL_HORIZON,
                 "max_coefficients": MAX_COEFFICIENTS, "qat_tokens": QAT_TOKENS, "qat_seq_len": QAT_SEQ_LEN,
                 "qat_accum": QAT_ACCUM, "qat_lr": QAT_LR, "val_seqs": VAL_SEQS, "eval_every": EVAL_EVERY,
                 "eval_limit": EVAL_LIMIT, "variant_flags": VARIANT_FLAGS[VARIANT]},
    "train_command": " ".join(map(str, train_cmd)),
    "artifacts": {p.name: {"bytes": p.stat().st_size, "sha256": sha256_file(p)} for p in artifacts},
    "kernel": {"correctness_passed": STATUS.get("kernel_tests", {}).get("ok"), "speed_claim_supported": speed_claim_ok},
    "caveats": caveats,
}
(OUT / "real_run_manifest.json").write_text(json.dumps(manifest, indent=2))
print("all stages ok:", manifest["all_stages_ok"])
for k, v in manifest["stages"].items(): print(f"  {k:<20} ok={v['ok']}  {v['minutes']} min" + ("  (watchdog)" if v["watchdog_ended_process"] else ""))
print("\ncaveats to state when you report this run:"); [print("  -", c) for c in caveats]


In [ ]:
# ===== 12. Package, copy to Drive, download =====
bundle = shutil.make_archive(str(OUT.parent / RUN_NAME), "zip", OUT)          # results only, no checkpoint
print("bundle:", bundle, f"{os.path.getsize(bundle) / 1e6:.1f} MB")
if SAVE_TO_DRIVE and IN_COLAB:
    dst = pathlib.Path(DRIVE_DIR) / RUN_NAME
    shutil.copytree(OUT, dst, dirs_exist_ok=True)
    print("copied to Drive:", dst)
if IN_COLAB:
    files.download(bundle)
